#### Objetivo: Testar como a diferenciação das ondas pode afetar a qualidade dos resultados e os testes do modelo

In [ ]:
# ---------------------
# Incluindo Bibliotecas
# ---------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VARMAX
from statsmodels.tsa.stattools import coint,kpss
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from itertools import product,combinations

#### Lendo Dados e Definindo Funções de Teste

In [ ]:
# ----------------------------------
# Função de detecção de cointegração
# ----------------------------------
def detect_cointegration(df,significance=0.05,verbose=False) -> list:
    """
    Testa cointegração par a par entre todas as séries.
    Retorna True se qualquer par for cointegrado.
    """
    cols = df.columns.tolist()
    cointegrated_pairs = []

    for c1, c2 in combinations(cols, 2):
        s1 = df[c1].dropna().astype(float)
        s2 = df[c2].dropna().astype(float)
        idx = s1.index.intersection(s2.index)
        try:
            _, p_value, _ = coint(s1.loc[idx], s2.loc[idx])
            if p_value < significance:
                cointegrated_pairs.append((c1, c2, p_value))
                if verbose:
                    print(f"  Cointegrados: {c1} x {c2} | p={p_value:.4f}")
        except Exception:
            pass

    return cointegrated_pairs

In [ ]:
# ----------------------------------------------------
# Função de verificação de estácionaridade kpss_test()
# ----------------------------------------------------
def kpss_test(serie,verbose=False) -> bool:
  kpss_serie = kpss(serie)

  if(verbose):
    print(f"  Test statitics  : {kpss_serie[0]}")
    print(f"  Test p_value    : {kpss_serie[1]}")
    print(f"  Number of lags  : {kpss_serie[2]}")
    print(f"  Critical values :")

    for key,val in kpss_serie[3].items():
      print(f"    {key} : {val}")

  return kpss_serie

In [ ]:
def detect_d_varmax(df_train, seasonal_period=None, verbose=False):
    """
    Detecta d e D para cada série do VARMAX.
    Retorna dict com d por coluna, D comum, e pares cointegrados.
    """
    d_dict = {}
    D_dict = {}

    for col in df_train.columns:
        serie = df_train[col].dropna().astype(float)
        d, D = detect_d_D(serie, seasonal_period=seasonal_period, verbose=False)
        d_dict[col] = d
        D_dict[col] = D
        if verbose:
            print(f"  {col}: d={d}, D={D}")

    # Verificar cointegração apenas entre séries I(1)
    cols_i1 = [c for c, v in d_dict.items() if v == 1]
    coint_pairs = []
    if len(cols_i1) >= 2:
        if verbose:
            print(f"\nTestando cointegração entre séries I(1): {cols_i1}")
        coint_pairs = detect_cointegration(
            df_train[cols_i1],
            verbose=verbose
        )

    # d comum conservador (máximo entre as séries)
    d_common = max(d_dict.values()) if d_dict else 0
    D_common = max(D_dict.values()) if D_dict else 0

    if verbose:
        print(f"\nd comum: {d_common} | D comum: {D_common}")
        if coint_pairs:
            print(f"⚠️  {len(coint_pairs)} par(es) cointegrado(s) — considere VECM")
        else:
            print("✓ Sem cointegração detectada — VARMAX com diferenciação é adequado")

    return {
        "d_by_col": d_dict,
        "D_by_col": D_dict,
        "d_common": d_common,
        "D_common": D_common,
        "cointegrated_pairs": coint_pairs
    }

In [ ]:
data = "dadosIniciais.xlsx"
data = pd.read_excel(data)

In [ ]:
df_input = pd.DataFrame()
df_input["Xreal"] = data.iloc[:,0]
df_input["Ximg"] = data.iloc[:,1]

In [ ]:
df_output = pd.DataFrame()
df_output["Yreal"] = data.iloc[:,2]
df_output["Yimg"] = data.iloc[:,3]

In [ ]:
df_concat = pd.concat([df_output,df_input],axis=1)

#### Verificação propriedades de estácionariadade

In [ ]:
yreal_serie = df_output["Yreal"].dropna().astype(float)
yimg_serie = df_output["Yimg"].dropna().astype(float)

print(f"Applying KPSS on Yreal Serie:")
kpss_test(yreal_serie,verbose=True)

print(f"\nApplying KPSS on Yimg Serie:")
kpss_test(yimg_serie,verbose=True)